# GCS Explorer

Use this notebook to:
- Detect credentials and project ID
- List available GCS buckets
- Browse objects inside a bucket (with optional prefix filters)

Note:
- If `GOOGLE_APPLICATION_CREDENTIALS` is set to a directory, the notebook will search for the first `.json` key inside it.
- If not set, it will search the project's `secrets/` folder for a `.json` key.
- You can override any detected values by editing variables in the cells below.


In [1]:
# Optional: ensure required packages are available in this environment
import sys, subprocess

def ensure(pkg: str):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# Core GCS libs
ensure("google.cloud.storage")
ensure("google.oauth2")

print("Dependencies ready.")


  Using cached google_crc32c-1.7.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (2.3 kB)
Using cached google_crc32c-1.7.1-cp312-cp312-macosx_12_0_arm64.whl (30 kB)
Using cached google_resumable_media-2.7.2-py2.py3-none-any.whl (81 kB)
Using cached proto_plus-1.26.1-py3-none-any.whl (50 kB)
Using cached rsa-4.9.1-py3-none-any.whl (34 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [google.cloud.storage]gle-api-core]
Dependencies ready.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, json, glob
from typing import Optional
from google.cloud import storage
from google.oauth2 import service_account

# You may override these
DEFAULT_SECRETS_DIR = "/Users/xuan/Desktop/Harvard-MIT-Course/AC215/AC215-Project-FitAI/secrets"
GOOGLE_APPLICATION_CREDENTIALS = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

key_path: Optional[str] = None

# 1) If env var points to a file, use it directly
if GOOGLE_APPLICATION_CREDENTIALS and os.path.isfile(GOOGLE_APPLICATION_CREDENTIALS):
    key_path = GOOGLE_APPLICATION_CREDENTIALS
# 2) If env var points to a directory, pick the first .json file inside
elif GOOGLE_APPLICATION_CREDENTIALS and os.path.isdir(GOOGLE_APPLICATION_CREDENTIALS):
    candidates = sorted(glob.glob(os.path.join(GOOGLE_APPLICATION_CREDENTIALS, "*.json")))
    key_path = candidates[0] if candidates else None
# 3) Else, search the project's secrets dir for a .json key
else:
    candidates = sorted(glob.glob(os.path.join(DEFAULT_SECRETS_DIR, "**", "*.json"), recursive=True))
    key_path = candidates[0] if candidates else None

credentials = None
project_id = os.getenv("GCP_PROJECT")

if key_path:
    credentials = service_account.Credentials.from_service_account_file(key_path)
    try:
        with open(key_path, "r") as f:
            d = json.load(f)
            project_id = project_id or d.get("project_id")
    except Exception:
        pass

print({
    "detected_key_path": key_path,
    "detected_project_id": project_id,
})


{'detected_key_path': '/Users/xuan/Desktop/Harvard-MIT-Course/AC215/AC215-Project-FitAI/secrets/rich-access-471117-r0-f17d92fbf298.json', 'detected_project_id': 'rich-access-471117-r0'}


In [3]:
# Initialize client
if credentials and project_id:
    storage_client = storage.Client(project=project_id, credentials=credentials)
elif project_id:
    storage_client = storage.Client(project=project_id)
else:
    storage_client = storage.Client()  # fall back to default ADC

print("Client initialized.")


Client initialized.


In [6]:
# List buckets in the detected/selected project
try:
    buckets = list(storage_client.list_buckets(project=project_id)) if project_id else list(storage_client.list_buckets())
    print(f"Found {len(buckets)} bucket(s):")
    for b in buckets:
        # Some fields may require an extra request; handle gracefully
        name = getattr(b, "name", None)
        location = getattr(b, "location", None)
        storage_class = getattr(b, "storage_class", None)
        print(f"- {name}  (location={location}, class={storage_class})")
except Exception as e:
    print("Error listing buckets:", e)


Error listing buckets: 403 GET https://storage.googleapis.com/storage/v1/b?project=rich-access-471117-r0&projection=noAcl&prettyPrint=false: sa-557@rich-access-471117-r0.iam.gserviceaccount.com does not have storage.buckets.list access to the Google Cloud project. Permission 'storage.buckets.list' denied on resource (or it may not exist).


In [5]:
# Choose a bucket to browse
# If you know the name, set it here; otherwise it tries to pick a sensible default.
BUCKET_NAME = None  # e.g., "fitai-data-bucket"

try:
    if BUCKET_NAME is None:
        # Prefer a well-known bucket name if present
        known = [b for b in storage_client.list_buckets(project=project_id) if getattr(b, "name", "") == "fitai-data-bucket"]
        if known:
            BUCKET_NAME = known[0].name
        else:
            # Fallback to the first bucket in the project
            first = next(iter(storage_client.list_buckets(project=project_id)), None)
            BUCKET_NAME = first.name if first else None
except Exception:
    BUCKET_NAME = BUCKET_NAME or "fitai-data-bucket"

print({"bucket_to_use": BUCKET_NAME})


{'bucket_to_use': 'fitai-data-bucket'}


In [7]:
# List objects in the selected bucket (optionally filter by prefix)
PREFIX = ""                   # e.g., "raw-literature" or "processed-literature"
MAX_RESULTS = 50

if BUCKET_NAME is None:
    print("No bucket selected. Set BUCKET_NAME above.")
else:
    try:
        bucket = storage_client.bucket(BUCKET_NAME)
        blobs = bucket.list_blobs(prefix=PREFIX)
        count = 0
        for blob in blobs:
            print(f"{blob.name}\t{blob.size} bytes")
            count += 1
            if count >= MAX_RESULTS:
                break
        print(f"Listed {count} object(s) (prefix='{PREFIX}')")
    except Exception as e:
        print("Error listing objects:", e)


processed-literature/	0 bytes
processed-literature/1-s2.0-S2666506924000178-main.txt	31608 bytes
processed-literature/Eccentric Exercise for Achilles Tendinopathy- A Narrative Review and Clinical Decision-Making Considerations.txt	54694 bytes
processed-literature/Effectiveness of specific scapular therapeutic exercises in patients with shoulder pain- a systematic review with meta-analysis.txt	83727 bytes
processed-literature/Evidence‑Based High‑Loading Tendon Exercise for 12 Weeks Leads to Increased Tendon Stiffness and Cross‑Sectional Area in Achilles Tendinopathy- A Controlled Clinical Trial.txt	93463 bytes
processed-literature/Nutrition_for_optimising_immune_function_and_recovery_from_injury_in_sports.txt	72944 bytes
processed-literature/Nutritional_Strategies_to_Improve_Post_exercise Recovery&Subsequent_Exercise_Performance_A Narrative_Review.txt	104003 bytes
processed-literature/Polyphenols_and_post_exercise_muscle_damage.txt	97454 bytes
processed-literature/Progression Models in 

In [ ]:
curl -X POST "http://localhost:8002/process-gcs"   -H "Content-Type: application/json"   -d '{
    "bucket_name": "fitai-data-bucket",
    "folder_path": "fitness-docs/processed-literature",
    "method": "char-split"
  }'